In [1]:
from requests import Response
import math
import load_TU_data
import pandas as pd
import geopandas as gpd
import requests
import importlib
from otp_client import get_stops_by_bbox_query

In [2]:
# print("Loading TU data...")
# tu_session, tu_tur, tu_deltur = load_TU_data.load_tu(
#     data_dir="/home/simpal/O/TU_Rejseplan/Data/TU/",
#     session_file="tu_session_secret_2015_2025.xlsx",
#     tur_file="tu_tur_secret_2015_2025.xlsx",
#     deltur_file="tu_deltur_2015_2025.xlsx"
# )
# print("TU data loaded")

In [3]:
data_dir = "/home/simpal/O/TU_Rejseplan/Data/TU/"
tu_stations = pd.read_excel(data_dir + "Stationer_tudatabase.xlsx")

In [4]:
# --- Add Lat/Lon (destination) ---
gdf_dest = gpd.GeoDataFrame(
    tu_stations,
    geometry=gpd.points_from_xy(tu_stations["e"], tu_stations["n"]),
    crs="EPSG:32632"
)
gdf_dest = gdf_dest.to_crs("EPSG:4326")

tu_stations["lon"] = gdf_dest.geometry.x
tu_stations["lat"] = gdf_dest.geometry.y

In [5]:
def get_response(url, query, variables):
    response = requests.post(
        url,
        json={
            "query": query,
            "variables": variables
        }
    )
    if response.status_code != 200:
        raise Exception(response.text)
    if "errors" in response.json():
        raise Exception(response.json()["errors"])
    return response


In [6]:
def parse_stops_to_df(response_data):
    response_data = response_data.json()
    # Retrieve the list of edges
    edges = response_data.get("data", {}).get("stopsByRadius", {}).get("edges", [])

    rows = []
    for edge in edges:
        node = edge.get("node", {})
        distance = node.get("distance")
        stop = node.get("stop", {})

        # Flatten routes details into formatted strings
        routes = stop.get("routes", [])
        route_modes = [r.get("mode") for r in routes if r.get("mode")]
        route_names = [r.get("shortName") for r in routes if r.get("shortName")]

        rows.append({
            "distance": distance,
            "stop_gtfsId": stop.get("gtfsId"),
            "name": stop.get("name"),
            "lat": stop.get("lat"),
            "lon": stop.get("lon"),
            "modes": ", ".join(set(route_modes)),
            "routes": ", ".join(route_names)
        })

    return pd.DataFrame(rows)

In [7]:
tu_stations.iloc[213]

statnavn                                                     Ishøj
stog                                                             1
metro                                                            0
andettog                                                         0
e                                                         711423.4
n                                                       6168100.73
uniklinie                                                      NaN
statmidte           0x00000000010CCDCCCCCCFEB52541EC51B82E89875741
aktivflag                                                        1
OpenDate                                                       NaN
ClosedDate                                                     NaN
GeoGraphicalZone                                            183012
komment                                                        NaN
letbane                                                        NaN
lon                                                      12.35

In [8]:
station_lat = tu_stations.iloc[212]["lat"]
station_lon = tu_stations.iloc[212]["lon"]

In [9]:
print(station_lat, station_lon)

56.14436436557575 9.159882905126809


In [10]:
response_data = get_stops_by_bbox_query(
    lat=56.14455 ,
    lon=9.158870,
    bbox_buffer_m=1000
)

Exception: [{'message': "Validation error (FieldUndefined@[stopsByBbox/edges]) : Field 'edges' in type 'Stop' is undefined", 'locations': [{'line': 9, 'column': 9}], 'extensions': {'classification': 'ValidationError'}}]

In [ ]:
parse_stops_to_df(response_data)

In [ ]:
from typing import Dict
from rapidfuzz import fuzz
import re

In [ ]:
TU_MODE_TO_GTFS = {
    "stog": "S_TRAIN",
    "metro":  "SUBWAY",
    "andettog":    "RAIL",
    "letbane":    "TRAM",
}

def _normalise_name(name: str) -> str:
    """Lowercase, remove punctuation, collapse whitespace."""
    name = name.lower()
    name = re.sub(r"[^\w\søæå]", " ", name)  # keep Danish letters
    name = re.sub(r"\s+", " ", name).strip()
    return name

def find_gtfs_stations_for_tu_station(
    tu_station: pd.Series,
    gtfs_df: pd.DataFrame = None,
    bbox_buffer_m: int = 400,
    otp_url: str = "http://localhost:8080/otp/gtfs/v1",
    name_match_threshold: float = 0.6,
) -> Dict[str, pd.Series]:
    """
    Find matching GTFS station(s) for a single TU station row.

    A TU station may correspond to multiple GTFS stops when GTFS splits
    modes into separate stops, OR to a single stop that serves all modes.

    Parameters
    ----------
    tu_station : pd.Series
        One row from tu_stations with 'lat', 'lon', mode flags, and 'statnavn'.
    gtfs_df : pd.DataFrame, optional
        Candidate GTFS stops (from parse_stops_to_df). If None, fetches from OTP.
    bbox_buffer_m : int
        Search bbox_buffer_m in metres.
    otp_url : str
        OTP endpoint.
    name_match_threshold : float
        Minimum similarity score (0–100) to accept a name match. Default 0.6.

    Returns
    -------
    dict mapping GTFS mode string → matched stop (pd.Series).
    e.g. {"S_TRAIN": <stop row>, "SUBWAY": <stop row>}
    If both modes share one stop, the same stop appears under both keys.
    """
    # 1. Active modes for this TU station
    active_modes = [
        gtfs_mode
        for tu_col, gtfs_mode in TU_MODE_TO_GTFS.items()
        if tu_station.get(tu_col, 0) == 1
    ]

    if not active_modes:
        print(f"No active modes for {tu_station['statnavn']}")
        return {}

    # 2. Fetch nearby GTFS stops if not provided
    if gtfs_df is None or gtfs_df.empty:
        response = get_stops_by_bbox_query(
            lat=tu_station["lat"],
            lon=tu_station["lon"],
            bbox_buffer_m=bbox_buffer_m,
            otp_url=otp_url,
        )
        gtfs_df = parse_stops_to_df(response)

    # 3. Keep only stops that serve at least one relevant mode
    def stop_serves_mode(modes_str: str, mode: str) -> bool:
        return mode in [m.strip() for m in modes_str.split(",")]

    relevant_stops = gtfs_df[
        gtfs_df["modes"].apply(
            lambda m: any(stop_serves_mode(m, mode) for mode in active_modes)
        )
    ].copy()

    if relevant_stops.empty:
        print(f"No relevant stops found for {tu_station['statnavn']}")
        return {}

    tu_name = str(tu_station.get("statnavn", ""))

    # 4. For each mode, pick the closest stop with good name match
    result: dict[str, pd.Series] = {}

    for mode in active_modes:
        mode_stops = relevant_stops[
            relevant_stops["modes"].apply(lambda m: stop_serves_mode(m, mode))
        ].copy()

        if mode_stops.empty:
            print(f"No stops found for {mode} in {tu_station['statnavn']}")
            continue

        # Score by name similarity (token_sort_ratio handles word order differences)
        if tu_name:
            mode_stops["name_similarity"] = mode_stops["name"].apply(
                lambda n: fuzz.token_sort_ratio(_normalise_name(tu_name), _normalise_name(n))
            )
            name_filtered = mode_stops[mode_stops["name_similarity"] >= name_match_threshold]

            if not name_filtered.empty:
                print(f"  Found {len(name_filtered)} stops for {tu_name} in {mode}")
                mode_stops = name_filtered
            else:
                print(
                    f"  Warning: no name match (threshold={name_match_threshold}) "
                    f"for '{tu_name}' in {mode} stops, using distance only"
                )

        # Closest stop by distance
        best = mode_stops.loc[mode_stops["distance"].idxmin()]
        result[mode] = best

    return result

In [ ]:
matches = find_gtfs_stations_for_tu_station(
    tu_station=tu_stations.iloc[7],
    name_match_threshold=0.6,
)

for mode, stop in matches.items():
    print(f"{mode}: {stop['name']} (name similarity: {stop.get('name_similarity', 'N/A')})")

In [ ]:
# for i, row in tu_stations.iterrows():
#     print(i, row["statnavn"])
#     matches = find_gtfs_stations_for_tu_station(
#         tu_station=row,
#         name_match_threshold=0,
#         bbox_buffer_m=bbox_buffer_m,
#     )
#
#     for mode, stop in matches.items():
#         print(f"{mode}: {stop['name']} (name similarity: {stop.get('name_similarity', 'N/A')})")






In [ ]:
import tu_gtfs_stations_match
importlib.reload(tu_gtfs_stations_match)
from tu_gtfs_stations_match import match_tu_gtfs_stations
match_tu_gtfs_stations(tu_stations,
                       bbox_buffer_m=1000,
                       period=(19725, 20086),
                       name_match_threshold = 0.8)

In [ ]:
tu_stations.iloc[212]

In [ ]:
17040.0

In [ ]:
import pandas as pd

# 1. Parse and extract coordinate data safely as floats
stations_list = []
for idx, row in tu_stations.iterrows():
    try:
        e_val = float(row['e'])
        n_val = float(row['n'])
        stations_list.append({
            'name': row['statnavn'],
            'e': e_val,
            'n': n_val
        })
    except (ValueError, TypeError):
        # Safely skip any row with unparseable coordinates
        continue

# 2. Compute all pairwise distances
all_pairs = []
n_stations = len(stations_list)

for i in range(n_stations):
    for j in range(i + 1, n_stations):
        s1 = stations_list[i]
        s2 = stations_list[j]

        # Calculate standard 2D Euclidean distance in meters
        dx = s1['e'] - s2['e']
        dy = s1['n'] - s2['n']
        dist = (dx**2 + dy**2)**0.5

        # Exclude duplicate entries or exact overlaps (dist < 1 meter)
        if dist >= 1.0:
            all_pairs.append({
                'station1': s1['name'],
                'station2': s2['name'],
                'distance_meters': dist,
                's1_coords': (s1['e'], s1['n']),
                's2_coords': (s2['e'], s2['n'])
            })

# 3. Sort pairs by distance in ascending order and select the top 10
top_10 = sorted(all_pairs, key=lambda x: x['distance_meters'])[:10]

# 4. Display the top 10 as a neat DataFrame
top_10_df = pd.DataFrame(top_10)
top_10_df.index = top_10_df.index + 1 # Start list at 1 instead of 0
top_10_df